In [ ]:
# Importing libraries and configuration

import tempfile
from dataclasses import replace
from pathlib import Path

from PIL import Image
from ultralytics.data.utils import check_det_dataset

from object_detection.config.loader import load_config
from object_detection.detection.evaluation import pair_detections_with_boxes
from object_detection.detection.inference import Detector
from object_detection.identification.cropping import crop_detection
from object_detection.identification.identify import identify
from object_detection.identification.index import create_index
from object_detection.utils.labels import load_labelled_boxes

cfg = load_config()
identification = cfg.identification

dataset = check_det_dataset(str(cfg.data.source_yaml))
class_names = dataset["names"]
val_imgs = sorted(p for p in Path(dataset["val"]).glob("*") if p.is_file())

print(f"{len(val_imgs)} validation images, {len(class_names)} classes")
print(f"detector: {cfg.inference.weights.name} at conf {cfg.inference.conf}")
print(f"embedder: {identification.model}")

In [ ]:
# Running detection and identification over the validation set

detector = Detector(cfg.inference.weights, cfg.inference.conf)
index = create_index(identification)

results = []
missed_total = 0

for img_path in val_imgs:
    labelled = load_labelled_boxes(img_path)
    true_boxes = [box for _, box in labelled]
    class_of_box = {box: class_names[class_id] for class_id, box in labelled}
    with Image.open(img_path) as image:
        detections = detector.predict(image)
        identifications = identify(
            image, detections, index,
            padding=identification.crop_padding,
            min_size=identification.min_crop_size,
            unknown_threshold=0,
        )
    identity_of = {id(i.detection): i for i in identifications}
    pairs, missed = pair_detections_with_boxes(detections, true_boxes)
    missed_total += len(missed)
    for detection, true_box in pairs:
        identity = identity_of[id(detection)]
        results.append({
            "image": img_path.name,
            "true_name": class_of_box[true_box] if true_box is not None else None,
            "predicted_name": identity.object_name,
            "score": identity.match_score})

real = [row for row in results if row["true_name"] is not None]

print(f"{len(results)} detections: {len(real)} on labelled objects, {len(results) - len(real)} false, {missed_total} objects missed")

In [ ]:
# Evaluating accuracy per class

correct = {name: 0 for name in class_names.values()}
total = {name: 0 for name in class_names.values()}

for row in real:
    total[row["true_name"]] += 1
    if row["predicted_name"] == row["true_name"]:
        correct[row["true_name"]] += 1

print(f"{'class':<16}{'top-1':>8}{'crops':>8}")
for name in class_names.values():
    if total[name]:
        print(f"{name:<16}{correct[name] / total[name]:>8.2f}{total[name]:>8}")

found_objects = sum(total.values())
print(f"{'ALL':<16}{sum(correct.values()) / found_objects:>8.2f}{found_objects:>8}")

In [ ]:
# Evaluating false identifications

for row in real:
    if row["predicted_name"] != row["true_name"]:
        print(f"{row['image'][:40]}: {row['true_name']} identified as {row['predicted_name']} ({row['score']:.3f})")

In [ ]:
# Evaluating how the unknown threshold affects objects that are in the index

print(f"{'threshold':>10}{'named':>8}{'correct':>9}{'rejected':>10}{'accuracy':>10}")
for threshold in [round(0.1 * i, 2) for i in range(11)]:
    named = [row for row in real if row["score"] is not None and row["score"] >= threshold]
    correct_named = [row for row in named if row["predicted_name"] == row["true_name"]]
    accuracy = len(correct_named) / len(named) if named else 0.0
    print(f"{threshold:>10}{len(named):>8}{len(correct_named):>9}{len(real) - len(named):>10}{accuracy:>10.2f}")

In [ ]:
# Evaluating score ranges (correct matches vs. mistakes)

correct_scores = [row["score"] for row in real if row["predicted_name"] == row["true_name"]]
wrong_scores = [row["score"] for row in real if row["predicted_name"] != row["true_name"]]

print(f"correct:    min {min(correct_scores):.3f}, median {sorted(correct_scores)[len(correct_scores) // 2]:.3f}, max {max(correct_scores):.3f}")
print(f"wrong:      {[round(score, 3) for score in wrong_scores]}")

In [ ]:
# Building a temporaary index with one class left out

HELD_OUT = "watch"

held_out_config = replace(
    identification,
    qdrant_location=tempfile.mkdtemp(prefix="qdrant_holdout_"),
    collection=f"without_{HELD_OUT.replace(' ', '_')}",
)

train_imgs = sorted(p for p in Path(dataset["train"]).glob("*") if p.is_file())
holdout_index = create_index(held_out_config)

stored = 0
for img_path in train_imgs:
    with Image.open(img_path) as image:
        for class_id, bbox in load_labelled_boxes(img_path):
            name = class_names[class_id]
            if name == HELD_OUT:
                continue
            crop = crop_detection(image, bbox, identification.crop_padding, identification.min_crop_size)
            if crop is not None:
                holdout_index.add_references([crop], object_id=name, object_name=name)
                stored += 1

print(f"{stored} references stored, with {HELD_OUT!r} left out")

In [ ]:
# Evaluating scores for the held-out object vs. objects that are in the index

held_out_scores = []
known_scores = []

for img_path in val_imgs:
    with Image.open(img_path) as image:
        for class_id, bbox in load_labelled_boxes(img_path):
            crop = crop_detection(image, bbox, identification.crop_padding, identification.min_crop_size)
            if crop is None:
                continue
            match = holdout_index.search([crop], limit=1)[0][0]
            if class_names[class_id] == HELD_OUT:
                held_out_scores.append((match.object_name, match.score))
            else:
                known_scores.append(match.score)

print(f"{HELD_OUT} crops (not in the index) matched:")
for name, score in sorted(held_out_scores, key=lambda pair: -pair[1]):
    print(f"    {name:<16}{score:.3f}")

print(f"\nhighest score for an unknown object: {max(score for _, score in held_out_scores):.3f}")
print(f"lowest score for a known object: {min(known_scores):.3f}")

The best unknown score (0.595) is higher than the worst known score (0.584), so there is an overlap in scores. This means that there is no threshold that can reject every unknown watch. However, this still means that the 0.5 threshold is still among the best thresholds for this case. If, for instance, 0.6 is used instead, all unknown objects will be rejected, but some known objects will be lost.

It's also worth noting that all of the unknown matches are with perfume, and both the perfume and watch in the dataset are long, dark elongated objects. Perhaps it is a good idea to test with other objects of different shapes like glasses.